# 🚗 CAPVIA AI - RT-DETR v2-L Defect Detection Training (Google Colab)
---
This notebook trains the **RT-DETR v2 Large** paint defect detector on Google Colab GPU.

### 📋 Workflow:
1. **GPU Check** - Verify CUDA environment
2. **Install Dependencies** - Install PyTorch, Ultralytics, Albumentations, ONNX
3. **Upload Dataset** - Upload `ai_model.zip` (auto-extracted & verified)
4. **Train Model** - 1024px, AdamW, Cosine Annealing, Mixed Precision
5. **Validate** - mAP50, mAP50-95, Precision, Recall
6. **Export & Package** - Auto-generate `CAPVIA_TRAINED_MODEL.zip` and download


## Step 1: GPU Environment Check
Ensure you are running on a GPU runtime (**Runtime > Change runtime type > T4 GPU**).

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    raise SystemError('GPU not detected! Go to Runtime > Change runtime type and select GPU.')


## Step 2: Install Dependencies
Installs Ultralytics, ONNX, Albumentations, and visualization tools.

In [ ]:
!pip install -q 'ultralytics>=8.3.0' albumentations opencv-python onnx onnxslim pyyaml matplotlib seaborn pandas tqdm
print('✓ All dependencies successfully installed!')


## Step 3: Upload and Extract Dataset (`ai_model.zip`)
The notebook expects `ai_model.zip` containing `dataset/data.yaml` and `train/`, `valid/`, `test/` splits.

In [ ]:
import os, zipfile, glob, yaml
from pathlib import Path

zip_file = 'ai_model.zip'
if not os.path.exists(zip_file):
    drive_path = '/content/drive/MyDrive/ai_model.zip'
    if os.path.exists(drive_path):
        import shutil
        shutil.copy(drive_path, zip_file)
        print('Copied ai_model.zip from Google Drive!')
    else:
        print('Please upload your ai_model.zip file:')
        from google.colab import files
        uploaded = files.upload()

print('Extracting ai_model.zip...')
with zipfile.ZipFile(zip_file, 'r') as zf:
    zf.extractall('.')

possible_yamls = glob.glob('**/data.yaml', recursive=True)
if not possible_yamls:
    raise FileNotFoundError('data.yaml not found after extraction! Please verify ai_model.zip structure.')

data_yaml_path = os.path.abspath(possible_yamls[0])
dataset_dir = os.path.dirname(data_yaml_path)
print(f'✓ Detected dataset configuration at: {data_yaml_path}')

with open(data_yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['path'] = dataset_dir
with open(data_yaml_path, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

train_imgs = glob.glob(f'{dataset_dir}/train/images/*.*')
val_imgs = glob.glob(f'{dataset_dir}/valid/images/*.*')
test_imgs = glob.glob(f'{dataset_dir}/test/images/*.*')
print(f'Dataset Summary: {len(train_imgs)} train | {len(val_imgs)} val | {len(test_imgs)} test images.')


## Step 4: Train RT-DETR v2 Large Detector
We train **RT-DETR-L** with:
- Image size: `1024`
- Epochs: `150`
- Optimizer: `AdamW` (lr=1e-4)
- Scheduler: `Cosine Annealing`
- Precision: `AMP (FP16)` enabled
- Early Stopping: `20` epochs patience


In [ ]:
from ultralytics import RTDETR

print('Initializing RT-DETR v2-L architecture...')
model = RTDETR('rtdetr-l.pt')

results = model.train(
    data=data_yaml_path,
    epochs=150,
    imgsz=1024,
    batch=16,
    device=0,
    optimizer='AdamW',
    lr0=1e-4,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,
    amp=True,
    patience=20,
    save_period=5,
    project='capvia_training',
    name='rtdetr_l_experiment',
    exist_ok=True,
    verbose=True,
)
print('✓ Training complete!')


## Step 5: Evaluate & Validate Model
Calculate mAP50, mAP50-95, precision, and recall on the test dataset.

In [ ]:
val_metrics = model.val(data=data_yaml_path, imgsz=1024, split='test')
print('\n=================== VALIDATION RESULTS ===================')
print(f'mAP@50:      {val_metrics.box.map50:.4f}')
print(f'mAP@50-95:   {val_metrics.box.map:.4f}')
print(f'Precision:   {val_metrics.box.mp:.4f}')
print(f'Recall:      {val_metrics.box.mr:.4f}')
print('==========================================================\n')


## Step 6: Export ONNX & Package into `CAPVIA_TRAINED_MODEL.zip`
Packages `best.pt`, `best.onnx`, `training_metrics.json`, loss curves, and confusion matrices.

In [ ]:
import os, shutil, json, zipfile, glob
from ultralytics import RTDETR

output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

train_run_dir = 'capvia_training/rtdetr_l_experiment'
best_pt = f'{train_run_dir}/weights/best.pt'
if not os.path.exists(best_pt):
    pt_matches = glob.glob('**/best.pt', recursive=True)
    best_pt = pt_matches[0] if pt_matches else None

if not best_pt or not os.path.exists(best_pt):
    raise FileNotFoundError('best.pt not found!')

# 1. Copy best.pt
shutil.copy2(best_pt, f'{output_dir}/best.pt')
print(f'1. Staged best.pt from {best_pt}')

# 2. Export best.onnx
print('2. Exporting to ONNX format (imgsz=1024, opset=16)...')
best_model = RTDETR(best_pt)
exported_onnx = best_model.export(format='onnx', imgsz=1024, opset=16, dynamic=True)
if exported_onnx and os.path.exists(exported_onnx):
    shutil.copy2(exported_onnx, f'{output_dir}/best.onnx')
    print('   Exported best.onnx successfully.')

# 3. Collect charts
for chart in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    src_chart = f'{train_run_dir}/{chart}'
    target_name = 'loss_curve.png' if chart == 'results.png' else chart.lower()
    if os.path.exists(src_chart):
        shutil.copy2(src_chart, f'{output_dir}/{target_name}')

# 4. Write training metrics JSON
metrics_dict = {
    'model': 'RT-DETR v2-L',
    'image_size': 1024,
    'mAP50': round(float(val_metrics.box.map50), 4),
    'mAP50_95': round(float(val_metrics.box.map), 4),
    'precision': round(float(val_metrics.box.mp), 4),
    'recall': round(float(val_metrics.box.mr), 4),
}
with open(f'{output_dir}/training_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

# 5. Create CAPVIA_TRAINED_MODEL.zip
zip_filename = 'CAPVIA_TRAINED_MODEL.zip'
with zipfile.ZipFile(zip_filename, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(output_dir):
        full_path = os.path.join(output_dir, f)
        if os.path.isfile(full_path):
            zf.write(full_path, arcname=f)
            print(f'   Included in zip: {f}')

size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f'\n✓ Successfully built {zip_filename} ({size_mb:.2f} MB)!')


## Step 7: Download Trained Model
Download `CAPVIA_TRAINED_MODEL.zip` to your computer.
Then follow `docs/RECONNECT_MODEL.md` in your project to copy `best.pt` into `ai_model/weights/`.

In [ ]:
from google.colab import files
files.download('CAPVIA_TRAINED_MODEL.zip')
print('✓ Download prompt opened in your browser!')
